# I1 Input Enhancement

In [ ]:
from pathlib import Path
import copy
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import log_loss
from torch.utils.data import DataLoader, Dataset
from model import SimpleCNN
from data_pipeline import build_canonical_images, build_input_batch, build_input_image, compute_channel_statistics, IcebergDataset

REPO_ROOT = Path.cwd()

if REPO_ROOT.name == "src":
    REPO_ROOT = REPO_ROOT.parent

if not (REPO_ROOT/"README.md").exists():
    raise RuntimeError("Start Jupyter from the kaggle_model repository root.")

TRAIN_PATH = REPO_ROOT/"data"/"raw"/"train.json"
TEST_PATH = REPO_ROOT/"data"/"raw"/"test.json"
FOLD_PATH = REPO_ROOT/"artifacts"/"folds.csv"

# print("Repository root:", REPO_ROOT)
# print("Training data:", TRAIN_PATH)
# print("Test data:", TEST_PATH)
# print("Fold file:", FOLD_PATH)

In [ ]:
train_df = pd.read_json(TRAIN_PATH)
test_df = pd.read_json(TEST_PATH)

print("Training table:", train_df.shape)
print("Test table:", test_df.shape)
print("Training columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())

assert len(train_df) == 1604
assert len(test_df) == 8424
assert "is_iceberg" in train_df.columns
assert "is_iceberg" not in test_df.columns

In [ ]:
sample = train_df.iloc[0]

print("ID：", sample["id"])
print("label：", sample["is_iceberg"])
print("angle：", sample["inc_angle"])

print("band_1 number：", len(sample["band_1"]))
print("band_2 number：", len(sample["band_2"]))

In [ ]:
train_images = build_canonical_images(train_df)
test_images = build_canonical_images(test_df)

print("Training images:", train_images.shape, train_images.dtype)
print("Test images:", test_images.shape, test_images.dtype)

In [ ]:
train_ids = train_df["id"].astype(str).to_numpy()
test_ids = test_df["id"].astype(str).to_numpy()

targets = train_df["is_iceberg"].to_numpy(dtype=np.float32)

train_angle_raw = pd.to_numeric(train_df["inc_angle"],errors="coerce").to_numpy(dtype=np.float32)

test_angle_raw = pd.to_numeric(test_df["inc_angle"],errors="coerce").to_numpy(dtype=np.float32)

train_angle_missing = np.isnan(train_angle_raw)
test_angle_missing = np.isnan(test_angle_raw)

fold_df = pd.read_csv(FOLD_PATH)
fold_df["id"] = fold_df["id"].astype(str)


if set(fold_df["id"]) != set(train_ids):
    raise ValueError("Fold IDs do not match training IDs.")

fold_lookup = fold_df.set_index("id")
aligned_folds = fold_lookup.loc[train_ids]

fold_ids = aligned_folds["fold"].to_numpy(dtype=np.int64)
fold_targets = aligned_folds["target"].to_numpy(dtype=np.int64)

if not np.array_equal(targets, fold_targets):
    raise ValueError("Training labels do not match fold targets.")

print("Training IDs:", train_ids.shape)
print("Targets:", targets.shape)
print("Fold IDs:", fold_ids.shape)
print("Fold counts:", np.bincount(fold_ids))
print("Missing training angles:", train_angle_missing.sum())
print("Missing test angles:", test_angle_missing.sum())

In [ ]:
INPUT_CHANNEL_COUNTS = {
    "raw_2ch": 2,
    "mean_3ch": 3,
    "difference_3ch": 3
}

In [ ]:
for input_config in INPUT_CHANNEL_COUNTS:
    enhanced = build_input_batch(train_images[:4], input_config)

    print(
        input_config,
        enhanced.shape,
        enhanced.dtype,
        np.isfinite(enhanced).all()
    )

mean_images = build_input_batch(train_images[:4], "mean_3ch")

difference_images = build_input_batch(train_images[:4], "difference_3ch")

expected_mean = (train_images[:4, 0] + train_images[:4, 1]) / np.float32(2.0)

expected_difference = (train_images[:4, 0] - train_images[:4, 1])

assert np.allclose(mean_images[:, 2], expected_mean)
assert np.allclose(difference_images[:, 2], expected_difference)

print("Input enhancement formula checks passed.")

In [ ]:
def show_input_features(row_index):
    image = train_images[row_index]

    band_1 = image[0]
    band_2 = image[1]
    mean_channel = (band_1 + band_2) / np.float32(2.0)
    difference_channel = band_1 - band_2

    channels = [
        ("Band 1", band_1, "gray"),
        ("Band 2", band_2, "gray"),
        ("Mean", mean_channel, "gray"),
        ("Difference", difference_channel, "coolwarm")
    ]

    label = int(targets[row_index])
    label_name = "Iceberg" if label == 1 else "Ship"

    fig, axes = plt.subplots(1, 4, figsize=(16, 4),constrained_layout=True)

    for axis, (name, channel, color_map) in zip(axes, channels):
        plot = axis.imshow(channel, cmap=color_map, interpolation="nearest")
        axis.set_title(name)
        axis.axis("off")
        fig.colorbar(plot, ax=axis, fraction=0.046)

    fig.suptitle(f"{label_name} | ID: {train_ids[row_index]}")

    plt.show()


ship_index = int(np.flatnonzero(targets == 0)[0])
iceberg_index = int(np.flatnonzero(targets == 1)[0])

show_input_features(ship_index)
show_input_features(iceberg_index)

In [ ]:
example_train_mask = fold_ids != 0

for input_config in INPUT_CHANNEL_COUNTS:
    channel_mean, channel_std = compute_channel_statistics(train_images, example_train_mask, input_config)

    print(input_config)
    print("Mean:", channel_mean)
    print("Standard deviation:", channel_std)

In [ ]:
model_2ch = SimpleCNN(in_channels=2)
model_3ch = SimpleCNN(in_channels=3)

output_2ch = model_2ch(torch.zeros(4, 2, 75, 75))

output_3ch = model_3ch(torch.zeros(4, 3, 75, 75))

print("Two-channel output:", output_2ch.shape)
print("Three-channel output:", output_3ch.shape)

assert output_2ch.shape == (4, 1)
assert output_3ch.shape == (4, 1)

In [ ]:
# smoke_result = run_one_fold(
#     fold_number=0,
#     input_config="mean_3ch",
#     in_channels=3,
#     num_epochs=1
# )

# print("Smoke-test fold:", smoke_result["fold"])
# print("Best epoch:", smoke_result["best_epoch"])
# print("Fold Log Loss:", smoke_result["fold_log_loss"])
# print("Validation predictions:", smoke_result["validation_probabilities"].shape)
# print("Test predictions:", smoke_result["test_probabilities"].shape)
# print("Channel mean:", smoke_result["channel_mean"])
# print("Channel standard deviation:", smoke_result["channel_std"])

In [ ]:
summary_b0 = run_experiment(
    run_id="b0_raw_2ch_seed42",
    input_config="raw_2ch",
    in_channels=2
)

In [ ]:
summary_i1a = run_experiment(run_id="i1a_mean_3ch_seed42",
    input_config="mean_3ch",
    in_channels=3
)

In [ ]:
summary_i1b = run_experiment(
    run_id="i1b_difference_3ch_seed42",
    input_config="difference_3ch",
    in_channels=3
)

In [ ]:
comparison_df = pd.DataFrame([summary_b0, summary_i1a, summary_i1b])

comparison_df = comparison_df.sort_values("overall_oof_log_loss", ascending=True).reset_index(drop=True)

comparison_df

In [ ]:
metric_files = [
    METRIC_DIR/"folds_b0_raw_2ch_seed42.csv",
    METRIC_DIR/"folds_i1a_mean_3ch_seed42.csv",
    METRIC_DIR/"folds_i1b_difference_3ch_seed42.csv",
]

all_fold_metrics = pd.concat([pd.read_csv(path) for path in metric_files], ignore_index=True)

all_fold_metrics.pivot(index="fold", columns="input_config", values="fold_log_loss")

In [ ]:
best_run_id = comparison_df.loc[0, "run_id"]

best_test_path = (PREDICTION_DIR / f"test_{best_run_id}.csv")

best_test_df = pd.read_csv(best_test_path)

submission_df = best_test_df[["id", "prediction"]].rename(
    columns={"prediction": "is_iceberg"})

submission_path = (PREDICTION_DIR/f"submission_{best_run_id}.csv")

submission_df.to_csv(submission_path, index=False)

print("Best run:", best_run_id)
# print("Submission path:", submission_path)
print("Submission shape:", submission_df.shape)

# print(submission_df.head())

assert submission_df.shape == (8424, 2)
assert submission_df["id"].is_unique
assert submission_df["is_iceberg"].between(0, 1).all()
assert submission_df.notna().all().all()


In [ ]:
comparison_path = (METRIC_DIR / "comparison_i1.csv")
comparison_df.to_csv(comparison_path,index=False)

print("Comparison saved to:", comparison_path)